# Listas com restrições

Lembrando o exemplo da aula anterior:

In [1]:
from pydantic import BaseModel, PositiveInt, ValidationError

class Circle(BaseModel):
    center: tuple[int, int] = (0, 0)
    radius: PositiveInt = 1

O `center` só pode ser um ponto no espaço 2D.

E se quiséssemos aceitar um ponto 2D (coordenada de 2 elementos) **ou** 3D (3 elementos)?

Uma abordagem seria esta:

In [2]:
class Sphere(BaseModel):
    center: tuple[int, int] | tuple[int, int, int] = (0, 0)
    radius: PositiveInt = 1

Assim podemos definir uma "esfera" 2D (um círculo — o nome do modelo não é dos melhores):

In [3]:
print(f"Sphere(center=(1, 1), radius=10): {Sphere(center=(1, 1), radius=10)!r}")

Sphere(center=(1, 1), radius=10): Sphere(center=(1, 1), radius=10)


ou uma esfera 3D:

In [4]:
print(f"Sphere(center=(1, 2, 3), radius=5): {Sphere(center=(1, 2, 3), radius=5)!r}")

Sphere(center=(1, 2, 3), radius=5): Sphere(center=(1, 2, 3), radius=5)


Se tentarmos uma tupla 4D, recebemos uma exceção:

In [5]:
try:
    Sphere(center=(1, 2, 3, 4))
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 2 validation errors for Sphere
center.tuple[int, int]
  Tuple should have at most 2 items after validation, not 4 [type=too_long, input_value=(1, 2, 3, 4), input_type=tuple]
    For further information visit https://errors.pydantic.dev/2.13/v/too_long
center.tuple[int, int, int]
  Tuple should have at most 3 items after validation, not 4 [type=too_long, input_value=(1, 2, 3, 4), input_type=tuple]
    For further information visit https://errors.pydantic.dev/2.13/v/too_long


Recebemos a exceção, mas o relatório de erro é meio estranho e nada fácil de entender (um erro por membro da união).

Por ora, vamos ignorar que pedimos uma tupla e supor que uma lista serve.

Então podemos usar a lista com restrições do Pydantic, limitando a lista a no mínimo 2 e no máximo 3 elementos:

In [6]:
from pydantic import conlist


class Sphere(BaseModel):
    center: conlist(int, min_length=2, max_length=3) = [0, 0]
    radius: PositiveInt = 1

In [7]:
print(f"Sphere(center=(1, 2)): {Sphere(center=(1, 2))!r}")

Sphere(center=(1, 2)): Sphere(center=[1, 2], radius=1)


In [8]:
print(f"Sphere(center=[1, 2, 3]): {Sphere(center=[1, 2, 3])!r}")

Sphere(center=[1, 2, 3]): Sphere(center=[1, 2, 3], radius=1)


E a exceção quando `center` não tem o número de elementos exigido:

In [9]:
try:
    Sphere(center=(1, 2, 3, 4))
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Sphere
center
  List should have at most 3 items after validation, not 4 [type=too_long, input_value=(1, 2, 3, 4), input_type=tuple]
    For further information visit https://errors.pydantic.dev/2.13/v/too_long


In [10]:
try:
    Sphere(center=(1, ))
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Sphere
center
  List should have at least 2 items after validation, not 1 [type=too_short, input_value=(1,), input_type=tuple]
    For further information visit https://errors.pydantic.dev/2.13/v/too_short


A exceção é muito mais clara.

Repare, aliás, que o Pydantic converteu a tupla passada em `center` num objeto `list`.

In [11]:
from pydantic import PydanticUserError


try:
    class Model(BaseModel):
        unique_elements: conlist(int, min_length=1, max_length=10, unique_items=True)
except PydanticUserError as ex:
    print(f"{type(ex).__name__}: {ex}")

PydanticUserError: `unique_items` is removed, use `Set` instead(this feature is discussed in https://github.com/pydantic/pydantic-core/issues/296)

For further information visit https://errors.pydantic.dev/2.13/u/removed-kwargs


Na documentação do `conlist`, há uma restrição adicional possível: unicidade, que obrigaria todos os elementos a serem únicos.

A documentação dizia que o parâmetro estava **depreciado** (ou seja, não use em código novo) e, na época da gravação, ele já tinha sido
**removido** — como a célula acima mostra. Fica o alerta para todos nós: não use funcionalidade depreciada!

(E uma implicância do autor, que funciona também em português: é **depreciado** no sentido de *deprecated* — "obsoleto, a ser removido" —
e não de *depreciated*, perda de valor contábil.)

O autor discorda da recomendação de usar `Set`: um set de fato só contém elementos únicos, mas não é um tipo de **sequência** — se
quisermos uma sequência (como lista) de elementos únicos, teremos de achar outro caminho (e acharemos, quando chegarmos aos validadores
customizados).

> 💡 **Complemento:** a forma moderna, sem `conlist`, é `Annotated[list[int], Field(min_length=2, max_length=3)]` — veremos `Annotated`
> na seção 7. Equivalente:

In [12]:
from typing import Annotated
from pydantic import Field

class SphereAnnotated(BaseModel):
    center: Annotated[list[int], Field(min_length=2, max_length=3)] = [0, 0]
    radius: PositiveInt = 1

try:
    SphereAnnotated(center=[1, 2, 3, 4])
except ValidationError as ex:
    print(f'ex.errors()[0]["type"]: {ex.errors()[0]["type"]}')

ex.errors()[0]["type"]: too_long
